# Agent Runner Smoke Test [LangGraph]

In [29]:
from __future__ import annotations

import argparse
import json
import os
import sys
import time
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable, Dict, List, Optional, Protocol, Tuple, TypedDict, Union
# from dotenv import load_dotenv
# load_dotenv()
from langgraph.graph import END, StateGraph
from openai import AzureOpenAI

from agent_runner import (
    AGG_CSV_COLUMNS,
    END as END_MARKER,
    aggregate_agent_results,
    candidate_label_by_id,
    evaluate_aggregated_predictions,
    load_spl_records_from_file,
    parse_json_with_end_marker,
    validate_postcoord_with_mrcm,
    write_csv_rows,
    write_jsonl,
)
from VaxMapper.src.utils.dailymed import CONTRA_Loinc, extract_section
from VaxMapper.src.utils.hyb_mapper import (
    DEFAULT_ITEM_TERM_KEYS,
    get_cached_mapper_resources,
    retrieve_candidates_for_item as retrieve_candidates_for_item_hybrid,
)
from VaxMapper.src.utils.llm_prompt import (
    CONTRA_EXTRACT_SYSTEM_PROMPT,
    CONTRA_EXTRACT_USER_PROMPT,
    DIRECT_VERIFY_SYSTEM_PROMPT,
    ROUTE_OR_FILL_SYSTEM_PROMPT,
    build_direct_verify_user_prompt,
    build_route_or_fill_user_prompt,
    extract_contraindication_items,
)
from VaxMapper.src.utils.snomed_utils import (
    DEFAULT_PREFILTER_CONTENT_TYPE,
    filter_terms_by_attribute_range,
    load_prefilter_memberships,
    load_snomed_dataframes,
)


_PREFILTER_ATTR_RANGE_CACHE: Dict[str, Any] = {}
_PREFILTER_MEMBERSHIP_CACHE: Dict[str, Dict[str, Dict[int, bool]]] = {}
_PREFILTER_ECL_CACHE: Dict[Tuple[int, str], bool] = {}
os.environ['CUDA_VISIBLE_DEVICES'] = '4,5'

In [31]:
from langgraph_agent_runner import build_llm, ContraLangGraphAgent, AgentRunConfig

In [3]:
import torch
torch.cuda.empty_cache()
import gc
gc.collect()

0

In [5]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [6]:
from typing import Any, Callable, Dict, List, Optional, Protocol, Tuple, TypedDict, Union
def resolve_runner_cuda_visible_devices() -> Optional[str]:
    for env_name in ("RUNNER_CUDA_VISIBLE_DEVICES", "CUDA_VISIBLE_DEVICES", "HF_CUDA_VISIBLE_DEVICES"):
        value = os.environ.get(env_name, "").strip()
        if value:
            return value
    return None

In [7]:
visible_devices = resolve_runner_cuda_visible_devices()

In [8]:
visible_devices

'4,5'

In [3]:
spl_list = 'results/setid_100.txt'
spl_records = load_spl_records_from_file(spl_list)
spl_set_id = spl_records[0]["SPL_SET_ID"]
spl_record = {}

In [32]:
import os 
os.environ['CUDA_VISIBLE_DEVICES'] = "4,5"
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"  # optional for debugging

In [34]:
llm = build_llm('huggingface')

Loading checkpoint shards:   0%|          | 0/11 [00:00<?, ?it/s]

In [35]:
agent = ContraLangGraphAgent(llm=llm, cfg=AgentRunConfig())

In [8]:
type(agent), type(llm)

(langgraph_agent_runner.ContraLangGraphAgent,
 langgraph_agent_runner.HuggingFaceChatLLM)

In [ ]:
# result = agent.process_spl(spl_records[0])
# Dig into the internal functions of the agent to test them in isolation

In [ ]:
# Class ContraLangGraphAgent
#         self.llm = llm
#         self.cfg = cfg or AgentRunConfig()
#         self.observer = observer
#         self.item_graph = self._build_item_graph()
#         self.spl_graph = self._build_spl_graph()
# # def process_spl()
# init_state: ContraState = {"spl_record": dict(spl_record)}

In [9]:
# Node 01
# resolve_contra_section_node()
contra_section = extract_section(str(spl_set_id), [CONTRA_Loinc])
section_payload = (contra_section.get("sections") or {}).get(CONTRA_Loinc, {})
spl_record["contra_section_text"] = section_payload.get("section_text") or ""
spl_record["contra_section_xml"] = section_payload.get("section_xml")
spl_record["product_name"] = contra_section.get("product_name")
spl_record["contra_section_found"] = bool(section_payload.get("section_text"))

In [10]:
spl_record["contra_section_text"]

'Administration with nitrates and nitric oxide donors (2.4,\n\n4.1)\n\nAdministration with guanylate cyclase (GC) stimulators, such as riociguat (2.4,\n\n4.2)'

In [39]:
# Node 02
# extract_items_node()
stop: List[str] = [END_MARKER]

In [ ]:
# 7-9 minutes
items, raw = extract_contraindication_items(llm.chat,spl_record.get("contra_section_text", ""),max_tokens=512,stop=stop,retries=2,)

In [12]:
items

[{'ci_text': 'Administration with nitrates',
  'contraindication_state_text': 'coadministration',
  'substance_text': 'nitrates',
  'severity_span': None,
  'course_span': None},
 {'ci_text': 'Administration with nitric oxide donors',
  'contraindication_state_text': 'coadministration',
  'substance_text': 'nitric oxide donors',
  'severity_span': None,
  'course_span': None},
 {'ci_text': 'Administration with guanylate cyclase (GC) stimulators, such as riociguat',
  'contraindication_state_text': 'coadministration',
  'substance_text': 'guanylate cyclase (GC) stimulators',
  'severity_span': None,
  'course_span': None}]

In [16]:
raw

'```json\n{"items":[{"ci_text": "Administration with nitrates", "contraindication_state_text": "coadministration", "substance_text": "nitrates", "severity_span": null, "clinical_course_span": null}, {"ci_text": "Administration with nitric oxide donors", "contraindication_state_text": "coadministration", "substance_text": "nitric oxide donors", "severity_span": null, "clinical_course_span": null}, {"ci_text": "Administration with guanylate cyclase (GC) stimulators, such as riociguat", "contraindication_state_text": "coadministration", "substance_text": "guanylate cyclase (GC) stimulators", "severity_span": null, "clinical_course_span": null}]}\n```'

In [ ]:
# Node 03 # prepare_item_node()
# Node 04: # process_item_node()
# result = self.item_graph.invoke(item_state)
# item_results = list(state.get("item_results", []))
# item_results.append(result["item_result"])

# retrieve_candidates_node()
resources = get_cached_mapper_resources()
retrieve_candidates_for_item()

In [2]:
from sentence_transformers import SentenceTransformer
import torch
st_model = SentenceTransformer("google/embeddinggemma-300m", device="cuda:0")
print(f"Success! Model is on: {st_model.device}")
from VaxMapper.src.utils.embedding_utils import maybe_move_index_to_gpu
import faiss
index_path = "FAISS/embeddinggemma-300m.bin"
cpu_index = faiss.read_index(index_path)
faiss_index = maybe_move_index_to_gpu(cpu_index)
from VaxMapper.src.utils.elastisearch_utils import get_es_client
from VaxMapper.src.utils.snomed_utils import load_snomed_dataframes
from VaxMapper.src.utils.hyb_mapper import build_es_index
snomed_source_dir = "snomed_us_source"
es = get_es_client()
snomed_frames = load_snomed_dataframes(snomed_source_dir=snomed_source_dir)
concept_meta_df = snomed_frames["concept_df"].set_index("conceptId")
es_index = "snomed_ct_es_index"
build_es_index(es,es_index,snomed_frames["snomed_complete_df"],rebuild_index=False,)

You are trying to use a model that was created with Sentence Transformers version 5.1.0, but you're currently using version 4.1.0. This might cause unexpected behavior or errors. In that case, try to update to the latest version.


Success! Model is on: cuda:0
Moving FAISS index to GPU (first visible device, FAISS device 0)...
Index moved to GPU successfully.
Index 'snomed_ct_es_index' already exists; skipping creation.


In [14]:
items = [{'ci_text': 'Administration with nitrates',
  'contraindication_state_text': 'coadministration',
  'substance_text': 'nitrates',
  'severity_span': None,
  'course_span': None},
 {'ci_text': 'Administration with nitric oxide donors',
  'contraindication_state_text': 'coadministration',
  'substance_text': 'nitric oxide donors',
  'severity_span': None,
  'course_span': None},
 {'ci_text': 'Administration with guanylate cyclase (GC) stimulators',
  'contraindication_state_text': 'coadministration',
  'substance_text': 'guanylate cyclase (GC) stimulators',
  'severity_span': None,
  'course_span': None},
 {'ci_text': 'Administration with riociguat',
  'contraindication_state_text': 'coadministration',
  'substance_text': 'riociguat',
  'severity_span': None,
  'course_span': None}]

In [25]:
query = 'Administration with nitrates'

In [26]:
from VaxMapper.src.utils.search_utils import search_query
hits = search_query(
            query_text=query,
            model=st_model,
            faiss_index=faiss_index,
            concept_meta_df=concept_meta_df,
            es=es,
            bm25_index=es_index,
            label_column="term",
            bm25_text_field="all_terms",
            bm25_id_field="conceptId",
            bm25_label_field="preferredTerm",
            k_dense=50,
            k_bm25=50,
            k_final=20,
            normalize_query=True,
        )

In [27]:
hits

[{'id': 89119000,
  'label': 'Nitrate salt (substance)',
  'fused': 0.027623285131627734,
  'from': 'bm25,dense'},
 {'id': 116865006,
  'label': 'Administration of albumin (procedure)',
  'fused': 0.026132699813337858,
  'from': 'bm25,dense'},
 {'id': 18629005,
  'label': 'Administration of drug or medicament (procedure)',
  'fused': 0.02533373786407767,
  'from': 'bm25,dense'},
 {'id': 37750001,
  'label': 'Administration of tranquilizer (procedure)',
  'fused': 0.023284147827264113,
  'from': 'bm25,dense'},
 {'id': 432955007,
  'label': 'Administration of sulfonylurea (procedure)',
  'fused': 0.02127659574468085,
  'from': 'bm25,dense'},
 {'id': 225948006,
  'label': 'Administration of laxative (procedure)',
  'fused': 0.020101010101010102,
  'from': 'bm25,dense'},
 {'id': 52685006,
  'label': 'Administration of analgesic (procedure)',
  'fused': 0.02000800320128051,
  'from': 'bm25,dense'},
 {'id': 275844006,
  'label': 'Gamma globulin administration (procedure)',
  'fused': 0.01852

#ToDo: Nice to have: A semantic type column in BM25 and FAISS that gives an option to filter before searching
Eg: 
* pregnancy -> filter through findings / disorder 
* administration with nitrates -> procedure 
* hypersensitivity -> disorder / finding 


In [ ]:
# Node 05: # direct_match_node()
# direct = self._verify_direct_match(
#                 state["item"].get("ci_text", ""),
#                 state["candidates"].get("direct_candidates", state["candidates"].get("focus_candidates", [])),)

In [36]:
ci_text = "Administration with nitrates"
user = build_direct_verify_user_prompt(ci_text, hits, max_n=10)

In [40]:
raw = llm.chat(
    [{"role": "system", "content": DIRECT_VERIFY_SYSTEM_PROMPT}, {"role": "user", "content": user}],
            max_tokens=384,
            temperature=1.0,
            stop=stop,
        )

In [41]:
raw

'{"direct_match":false,"selected_id":"N/A","selected_term":"N/A"}'

In [ ]:
# Node 06: # route_after_direct_match()
# if direct match --> assemble_direct_node()
# else prefilter_node

In [ ]:
# Node 07: prefilter_node()
# prefilter_slot_candidates()